In [29]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

m.centerObject(panama_geom, 7)

# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

# Create a single DEM image and clip it
elevation = (
    dataset
    .select('DEM')
    .first()
    .clip(panama_geom)
)

# Construct a collection of corresponding Dynamic World and Sentinel-2 for
# inspection. Filter by region and date.

START = "2024-01-01"
END = "2024-12-31"

dw_col = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(panama_geom)
    .filterDate(START, END)
)

s2_col = ee.ImageCollection('COPERNICUS/S2_HARMONIZED');

# Link DW and S2 source images.
linked_col = dw_col.linkCollection(s2_col, s2_col.first().bandNames());

# Get example DW image with linked S2 image.
linked_image = ee.Image(linked_col.first())

# Create a visualization that blends DW class label with probability.
# Define list pairs of DW LULC label and color.
CLASS_NAMES = [
    'water',
    'trees',
    'grass',
    'flooded_vegetation',
    'crops',
    'shrub_and_scrub',
    'built',
    'bare',
    'snow_and_ice',
]

VIS_PALETTE = [
    '419bdf',
    '397d49',
    '88b053',
    '7a87c6',
    'e49635',
    'dfc35a',
    'c4281b',
    'a59b8f',
    'b39fe1',
]

# Create an RGB image of the label (most likely class) on [0, 1].
#dw_rgb = (
    #linked_image.select('label')
    #.visualize(min=0, max=8, palette=VIS_PALETTE)
    #.divide(255)
#)

# Get the most likely class probability.
#top1_prob = linked_image.select(CLASS_NAMES).reduce(ee.Reducer.max())

# Create a hillshade of the most likely class probability on [0, 1]
#top1_prob_hillshade = ee.Terrain.hillshade(top1_prob.multiply(100)).divide(255)

# Combine the RGB image with the hillshade.
#dw_rgb_hillshade = dw_rgb.multiply(top1_prob_hillshade)

dw_reprojected = linked_image.select('label').reproject(
    crs = elevation.projection(), scale = linked_image.projection().nominalScale()
)

# Display the Dynamic World visualization with the source Sentinel-2 image.
m = geemap.Map()

# m.set_center(8.535, -80.100, 8)

m.addLayer(
    linked_image.clip(panama_geom),
    {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']},
    'Sentinel-2 L1C',
)

m.addLayer(
    dw_reprojected.select('label').clip(panama_geom), 
    {"min": 0, "max": 8, "palette": VIS_PALETTE},
    "Dynamic World Reprojected"
)

#m.add_layer(
    #dw_rgb_hillshade,
    #{'min': 0, 'max': 0.65},
    #'Dynamic World V1 - label hillshade',
#)

m

EEException: Image.projection: The bands of the specified image contains different projections. Use Image.select to pick a single band.

In [34]:
linked_image.getInfo()

{'type': 'Image',
 'bands': [{'id': 'water',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'dimensions': [4208, 10942],
   'crs': 'EPSG:32617',
   'crs_transform': [10, 0, 267490, 0, 10, 790400]},
  {'id': 'trees',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'dimensions': [4208, 10942],
   'crs': 'EPSG:32617',
   'crs_transform': [10, 0, 267490, 0, 10, 790400]},
  {'id': 'grass',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'dimensions': [4208, 10942],
   'crs': 'EPSG:32617',
   'crs_transform': [10, 0, 267490, 0, 10, 790400]},
  {'id': 'flooded_vegetation',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'dimensions': [4208, 10942],
   'crs': 'EPSG:32617',
   'crs_transform': [10, 0, 267490, 0, 10, 790400]},
  {'id': 'crops',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'dimensions': [4208, 10942],
   'crs': 'EPSG:32617',
   'crs_transform': [10, 0, 267490, 0, 10, 790400]},
  {'id': 'shrub

In [ ]:
dw_reprojected.select('label').getInfo()

EEException: Image.projection: The bands of the specified image contains different projections. Use Image.select to pick a single band.

In [ ]:
linked_image.select('label').projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:32634',
 'transform': [10, 0, 400150, 0, 10, 5790450]}